<a href="https://colab.research.google.com/github/Echo9k/3-potential_talents/blob/main/Potential%20Talent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Proyecto de NLP: Detección y Clasificación de Talento

#### **Contexto**

Como empresa de reclutamiento y gestión de talento, nuestro objetivo es encontrar personas altamente capacitadas para cubrir roles en compañías tecnológicas. Esta tarea es desafiante por varias razones:

1. **Comprensión del rol**: Para encontrar al candidato ideal, debemos entender muy bien las necesidades y expectativas del cliente para el puesto en cuestión.
2. **Identificación de habilidades clave**: Es esencial reconocer las cualidades que hacen destacar a un candidato específico en cada rol.
3. **Ubicación del talento**: Encontrar fuentes donde se concentran estos candidatos también es un reto.

El proceso actual implica un gran esfuerzo manual. Para automatizar y optimizar este trabajo, buscamos desarrollar una solución que ahorre tiempo y nos permita identificar candidatos potenciales de manera más precisa. Con el tiempo, queremos construir un sistema de aprendizaje automático que no solo encuentre a los candidatos, sino que también los clasifique en función de qué tan bien se ajustan al rol.

Actualmente, estamos en una etapa en la que seleccionamos candidatos de manera semi-automatizada. Por lo tanto, el primer enfoque será clasificar a los candidatos en función de su ajuste para el puesto. Este proceso se basa en la búsqueda de palabras clave como "ingeniero de software full-stack", "gerente de ingeniería" o "aspirante a recursos humanos", y las palabras clave cambiarán según el rol.

Después de listar y clasificar a los candidatos, realizamos una **revisión manual** para evaluar qué tan bien encajan en el rol. Durante esta revisión, podríamos decidir que el candidato más adecuado no sea el primero de la lista, sino quizás el séptimo. En este caso, es importante que el sistema pueda **reordenar** la lista basándose en esta señal supervisada (al "marcar" a este séptimo candidato como ideal para el rol). Esperamos que cada vez que marquemos a un candidato, la lista se reorganice para reflejar mejor las preferencias para el puesto.




#### **Descripción de los Datos**

La información de los candidatos proviene de nuestros procesos de selección y ha sido anonimizada para proteger su privacidad. A cada candidato se le asignó un identificador único.

- **id**: Identificador único del candidato (*numérico*).
- **job_title**: Título del trabajo del candidato (*texto*).
- **location**: Ubicación geográfica del candidato (*texto*).
- **connections**: Número de conexiones que tiene el candidato (500+ significa más de 500, *texto*).
- **fit**: Nivel de ajuste del candidato para el rol, entre 0 y 1 (*numérico, probabilidad*).
- **keywords**: Palabras clave asociadas como “aspirante a recursos humanos” o “buscando recursos humanos”.


#### **Objetivos del Proyecto**

- **Objetivo Principal**: Predecir el nivel de ajuste de los candidatos (variable *fit*) basado en su información.

- **Métricas de Éxito**:
  - Clasificar a los candidatos en función de su puntaje de ajuste (*fit*).
  - Reordenar la lista de candidatos cada vez que uno sea marcado como ideal.


#### **Desafíos Actuales**

1. **Desarrollo de un Algoritmo Robusto**: Queremos un algoritmo confiable. Necesitamos una explicación clara de cómo funciona la solución y cómo mejora la clasificación después de cada acción de marcado.

2. **Filtrado de Candidatos No Aptos**: ¿Cómo podemos filtrar a los candidatos que no deberían estar en la lista desde el principio?

3. **Definición de un Punto de Corte**: ¿Es posible establecer un umbral que funcione para otros roles sin perder a candidatos de alto potencial?

4. **Reducción del Sesgo Humano**: ¿Tienes ideas para automatizar aún más el proceso y reducir el sesgo humano en la selección?

In [ ]:
!pip install wget
!pip install sentence_transformers

In [ ]:
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("stopwords")
nltk.download('punkt_tab')  # punkt tab
wget.download('http://nlp.stanford.edu/data/glove.6B.zip')  # globe

#### This project is about how to predict fit the candidate is based on their available information.

### Importing Librariers

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
root_dir = "/content/drive/MyDrive/wdir/repos/Apziva/3-potential_talents/"

%cd {root_dir}
%pwd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/wdir/repos/Apziva/3-potential_talents


'/content/drive/MyDrive/wdir/repos/Apziva/3-potential_talents'

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("data/raw/data.csv")
df.head()

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN


In [ ]:
print(f"The data shape is {df.shape}")

The data shape is (104, 5)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          104 non-null    int64  
 1   job_title   104 non-null    object 
 2   location    104 non-null    object 
 3   connection  104 non-null    object 
 4   fit         0 non-null      float64
dtypes: float64(1), int64(1), object(3)
memory usage: 4.2+ KB


In [ ]:
# The **Fit** column is targeted column soe we can drop it.
df.drop("fit", axis=1, inplace=True)

In [ ]:
df.head()

,id,job_title,location,connection
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+


In [ ]:
### Checking the numbers of unique value in each columns
for col in df.columns:
    print(f"There is {df[col].nunique()} unique value is {col} column")

There is 104 unique value is id column
There is 52 unique value is job_title column
There is 41 unique value is location column
There is 33 unique value is connection column


In [ ]:
print("The number of duplicated row is:", df.duplicated().sum())


The number of duplicated row is: 0


In [ ]:
df_dup = df.drop(["id"], axis=1)
print("Number of duplicate entries:" , df_dup.duplicated().sum())

Number of duplicate entries: 51


In [ ]:
df[df_dup.duplicated(keep=False)]

,id,job_title,location,connection
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+
...,...,...,...,...
60,61,HR Senior Specialist,San Francisco Bay Area,500+
61,62,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500+
62,63,Student at Chapman University,"Lake Forest, California",2
63,64,"SVP, CHRO, Marketing & Communications, CSR Off...","Houston, Texas Area",500+


In [ ]:
## Dropping the duplicated rows
df_dup =df_dup.drop_duplicates()
df =pd.concat([df["id"], df_dup], axis=1).dropna(axis=0)
print("Shape fo non-duplicated dataframe:", df.shape)

Shape fo non-duplicated dataframe: (53, 4)


In [ ]:
df=df.reset_index(drop=True)
df.shape

(53, 4)

In [ ]:
df.job_title.value_counts()

,count
job_title,
Aspiring Human Resources Professional,2
2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional,1
Lead Official at Western Illinois University,1
Senior Human Resources Business Partner at Heil Environmental,1
Aspiring Human Resources Professional | An energetic and Team-Focused Leader,1
HR Manager at Endemol Shine North America,1
Human Resources professional for the world leader in GIS software,1
RRP Brand Portfolio Executive at JTI (Japan Tobacco International),1
Information Systems Specialist and Programmer with a love for data and organization.,1


In [ ]:
from collections import Counter

In [ ]:
## Checking most frequent words..
words_counts = Counter()
for i in range(len(df)):
    for word in df.job_title[i].split(" "):
        words_counts[word] += 1

print("Number of total words" , len(words_counts))
words_counts.most_common()

Number of total words 221


[('Human', 34),
 ('Resources', 29),
 ('at', 22),
 ('and', 13),
 ('Aspiring', 11),
 ('|', 10),
 ('Professional', 7),
 ('in', 6),
 ('University', 6),
 ('Seeking', 6),
 ('Business', 5),
 ('Student', 5),
 ('Generalist', 5),
 ('Manager', 5),
 ('of', 4),
 ('Specialist', 4),
 ('&', 4),
 ('Management', 4),
 ('seeking', 4),
 ('', 4),
 ('an', 3),
 ('Director', 3),
 ('for', 3),
 ('College', 2),
 ('aspiring', 2),
 ('professional', 2),
 ('Coordinator', 2),
 ('HR', 2),
 ('Senior', 2),
 ('internship', 2),
 ('Resources,', 2),
 ('Staffing', 2),
 ('North', 2),
 ('a', 2),
 ('Resources.', 2),
 ('Major', 2),
 ('to', 2),
 ('Information', 2),
 ('Systems', 2),
 ('-', 2),
 ('Position', 2),
 ('2019', 1),
 ('C.T.', 1),
 ('Bauer', 1),
 ('Graduate', 1),
 ('(Magna', 1),
 ('Cum', 1),
 ('Laude)', 1),
 ('Native', 1),
 ('English', 1),
 ('Teacher', 1),
 ('EPIK', 1),
 ('(English', 1),
 ('Program', 1),
 ('Korea)', 1),
 ('People', 1),
 ('Development', 1),
 ('Ryan', 1),
 ('Advisory', 1),
 ('Board', 1),
 ('Member', 1),
 ('Ce

In [ ]:
df.location.value_counts()

,count
location,
"Houston, Texas Area",4
"Raleigh-Durham, North Carolina Area",3
Greater New York City Area,3
"Austin, Texas Area",2
Amerika Birleşik Devletleri,2
Kanada,2
Greater Philadelphia Area,2
Greater Atlanta Area,2
"Torrance, California",1


### Cleaning the data

    * Removing special character
    * Writting the text in lower case
    * Removing stopwords and applying lemmatization
    

**Lemmatization usually refers to doing things properly with the use of a vocabulary and morphological analysis of words, normally aiming to remove inflectional endings only and to return the base or dictionary form of a word, which is known as the lemma .**

In [ ]:
## Removing special character
df = df.replace({'location' : { "[\'!#)$%&(*+-./:;<=>?@[\]^_`{|}~\n]" : " "}}, regex=True)
df = df.replace({'connection' : { "[\'!#)$%&(*+-./:;<=>?@[\]^_`{|}~\n]" : " "}}, regex=True)

In [ ]:
df.head()

,id,job_title,location,connection
0,1,2019 C.T. Bauer College of Business Graduate (...,Houston Texas,85
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500
2,3,Aspiring Human Resources Professional,Raleigh Durham North Carolina Area,44
3,4,People Development Coordinator at Ryan,Denton Texas,500
4,5,Advisory Board Member at Celal Bayar University,İzmir Türkiye,500


In [ ]:
df["job_title"] =df["job_title"].str.lower()
df["location"]=df["location"].str.lower()
df.head()

,id,job_title,location,connection
0,1,2019 c.t. bauer college of business graduate (...,houston texas,85
1,2,native english teacher at epik (english progra...,kanada,500
2,3,aspiring human resources professional,raleigh durham north carolina area,44
3,4,people development coordinator at ryan,denton texas,500
4,5,advisory board member at celal bayar university,i̇zmir türkiye,500


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.tokenize.treebank import TreebankWordDetokenizer

import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
## Removing stopwords and applaying lemmatization
def cleaning(df, col):
    stop_words =set(stopwords.words("english"))
    for i in range(len(df)):
        word_token =word_tokenize(df[col][i])
        tokens_without_sw = [w for w in word_token if w not in stop_words]
        lemmatized_sentence = []
        for word in tokens_without_sw:
            lemmatized_sentence.append(WordNetLemmatizer().lemmatize(word))
        df[col][i] =TreebankWordDetokenizer().detokenize(lemmatized_sentence)

In [ ]:
print("Job title before removing stopwords:\n", df.job_title.head(1))
print("-"*100)

# Removing stop words
cleaning(df, "job_title")

print("Job title after removing stopwords:\n", df.job_title.head(1))
print("-"*100)
df.head()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Job title before removing stopwords:
 0    2019 c.t. bauer college of business graduate (...
Name: job_title, dtype: object
----------------------------------------------------------------------------------------------------
Job title after removing stopwords:
 0    2019 c.t . bauer college business graduate (ma...
Name: job_title, dtype: object
----------------------------------------------------------------------------------------------------


,id,job_title,location,connection
0,1,2019 c.t . bauer college business graduate (ma...,houston texas,85
1,2,native english teacher epik (english program k...,kanada,500
2,3,aspiring human resource professional,raleigh durham north carolina area,44
3,4,people development coordinator ryan,denton texas,500
4,5,advisory board member celal bayar university,i̇zmir türkiye,500


In [ ]:
dir_replacements_job_title = { 'chro' : 'chief human resources officer', 'svp' : 'senior vice president'
        ,'gphr' : 'global professional in human resources','hris' : 'human resources management system'
        , 'csr' : 'corporate social responsibility', 'sphr' : 'strategic and policy-making certification'
        , 'hr' : 'human resources', "[\'!#)$%&(*+-./:;<=>?@[\]^_`{|}~\n]" : "", r'[0-9]' : '', 'epik' : 'tech'
        , 'styczynski lab' : 'tech', 'gi' : 'tech', "schwan's" : 'tech', 'ryan' : 'not tech'
        , 'engine' : 'not tech', 'buckhead atlanta' : 'not tech', 'loparex' : 'not tech', 'delphi Hardware' : 'not tech'
        , "jti" : 'not tech', 'traveler' : 'not tech', 'luxottica' : 'not tech'
        , 'beneteau' : 'not tech', 'scottMadden' : 'not tech', 'ey' : 'not tech', 'endemol' : 'not tech'
        , 'nortia staffing' : 'not tech', 'heil environmental' : 'not tech', 'excellence logging' : 'not tech'}

In [ ]:
## Replace the abbreviation with their description
df = df.replace({'job_title' : dir_replacements_job_title}
        , regex=True)

#### Word Embedding Techniques

I will be using 4 different strategies to find the similarties betwen the targeted sentences and each job title as follows

 1. **TF-IDF**(term frequency-inverse document frequency) is a statistical measure that evaluates how relevant a word is to a document in a collection of documents.

    
  2. **GloVe** stands for global vectors for word representation. It is used for generating word embeddings by aggregating global word-word co-occurrence matrix from a corpus. The resulting embeddings show interesting linear substructures of the word in vector space.
    
   
   3. **Word2Vec** is a technique which produces word embeddings for better word representation. We can also say it consists of models for generating word embedding which are shallow two layer neural networks having one input layer, one hidden layer and one output layer.
    
   
   4. **BERT** It is designed to pre-train deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context.

### 1. TF-IDF

In [ ]:
import tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.spatial.distance import cosine

In [ ]:
## Convert hob titles column into a list
job_title_list =list(df["job_title"])

# Vectorize job_title_list
vectorizer = TfidfVectorizer().fit(job_title_list)
X =vectorizer.transform(job_title_list)

# Get feature names in all the documents
# Use get_feature_names_out() instead of get_feature_names()
feature_names = vectorizer.get_feature_names_out()
print("Number of unique features: ", len(feature_names))
print("First 5 features: ", feature_names[:5])

# convert job titles into array
tfidf_vector =X.toarray()
print("Shape of Tfidf Vector: ", tfidf_vector.shape)

Number of unique features:  167
First 5 features:  ['administration' 'administrative' 'admission' 'advisory' 'always']
Shape of Tfidf Vector:  (53, 167)


In [ ]:
## Calculating the similarity
def similarity(phrase, column_name):
    # convert search phrase into a vectors
    X = vectorizer.transform(phrase)
    X_vectors = X.toarray()
    print("Shape of search phrase vector:", X_vectors.shape)

    # calculate Tfidf cosine similarity
    sim_score_list = []
    for x in range(len(df)):
        # Fix: Select the first row (assuming single phrase) of X_vectors
        # to make it 1-dimensional
        sim_score_list.append(1 - cosine(tfidf_vector[x], X_vectors[0]))

    df[column_name] =sim_score_list

In [ ]:
# Searched prhase
phrases = pd.DataFrame({"phrase":["aspiring human resources"]})
cleaning(phrases, "phrase")

In [ ]:
similarity([phrases.phrase[0]], "TF-IDF_fit_score")

Shape of search phrase vector: (1, 167)


**Let's see the top and lowest matching title job rows with the phrase (aspiring human resources).**

In [ ]:
df.sort_values(by ="TF-IDF_fit_score", ascending=False).head()

,id,job_title,location,connection,TF-IDF_fit_score
2,3,aspiring human resource professional,raleigh durham north carolina area,44,0.781184
45,97,aspiring human resource professional,kokomo indiana area,71,0.781184
5,6,aspiring human resource specialist,greater new york city area,1,0.679890
21,73,aspiring human resource manager seeking intern...,houston texas area,7,0.618600
12,27,aspiring human resource management student see...,houston texas area,500,0.442293


In [ ]:
df.sort_values(by ="TF-IDF_fit_score", ascending=False).tail()

,id,job_title,location,connection,TF-IDF_fit_score
34,86,information system specialist programmer love ...,gaithersburg maryland,4,0.0
33,85,rrp brand portfolio executive not tech japan t...,greater philadelphia area,500,0.0
28,80,junior me entechneer information system,myrtle beach south carolina area,52,0.0
1,2,native english teacher tech english program korea,kanada,500,0.0
52,104,director administration excellence logtechng,katy texas,500,0.0


### 2. GloVe

In [ ]:
import wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9656 sha256=5c9b95542d6344f643f93bd4da33311620e4d05902da7e27dc6e2e87dae17a13
  Stored in directory: /root/.cache/pip/wheels/8b/f1/7f/5c94f0a7a505ca1c81cd1d9208ae2064675d97582078e6c769
Successfully built wget


'glove.6B.zip'

In [ ]:
import zipfile as zf
files = zf.ZipFile("glove.6B.zip", 'r')
files.extractall('GloVe')
files.close()

In [ ]:
import gensim.downloader as api
from gensim.test.utils import get_tmpfile
from gensim.scripts.glove2word2vec import glove2word2vec
from gensim.models import KeyedVectors, Word2Vec

# Create temp file and save converted embedding into it
target_file = get_tmpfile("word2vec.6B.50d.txt")
glove2word2vec("GloVe/glove.6B.50d.txt", target_file)

# Load the converetd embedding into memory
glove_model = KeyedVectors.load_word2vec_format(target_file)

# Save as binary data
glove_model.save_word2vec_format('word2vec.6B.50d.bin.gz', binary=True)

In [ ]:
def doc_token_vectors(sentence, model, vector_dimensions):
    """
    Generate token vectors for a sentence using the provided model.

    Args:
        sentence (str): The input sentence.
        model (gensim.models.KeyedVectors): The word embedding model.
        vector_dimensions (int): The dimensionality of the word embeddings.

    Returns:
        np.ndarray: A 2D numpy array where each row represents a token vector.
    """
    word_tokens = word_tokenize(sentence)
    filtered_words = [w for w in word_tokens if w in model]
    sentence_vector_list = []
    for j in range(len(word_tokens)):
        if word_tokens[j] in filtered_words:
            token_vector = model[word_tokens[j]]
        else:
            token_vector = np.zeros(vector_dimensions)
        sentence_vector_list.append(token_vector)

    # Convert the list of token vectors to a NumPy array
    return np.array(sentence_vector_list)

In [ ]:
def model_fitt_score(df, col, model, vector_dimensions, phrase, fitt_col_name):
    """
    Calculate similarity scores between job titles and a search phrase using word embeddings.

    Args:
        df (pandas.DataFrame): The DataFrame containing job titles.
        col (str): The column name containing job titles.
        model (gensim.models.KeyedVectors): The word embedding model.
        vector_dimensions (int): The dimensionality of the word embeddings.
        phrase (str): The search phrase.
        fitt_col_name (str): The name of the column to store similarity scores.
    """
    # Vectorize job title using the model
    model_vectors = []
    for i in range(len(df)):
        # Call doc_token_vectors with the correct number of arguments
        # and assign the result to model_sentence_vector
        model_sentence_vector = doc_token_vectors(df[col][i], model, vector_dimensions)
        model_vectors.append(model_sentence_vector)

    # Vectorize searched phrase using the model
    model_search_phrase_vector = doc_token_vectors(phrase, model, vector_dimensions)

    # Calculate cosine similarity between searched phrase and job title
    model_similarity =[]
    for i in range(len(df)):
        sim_score = 1 - cosine(np.mean(model_vectors[i], axis = 0), np.mean(model_search_phrase_vector, axis =0))
        model_similarity.append(sim_score)

    # Add model similarity score to the pt dataframe
    df[fitt_col_name] = model_similarity

In [ ]:
model_fitt_score(df, 'job_title', glove_model, 50, phrases.phrase[0], 'GloVe_fit_score')

In [ ]:
df.sort_values(by="GloVe_fit_score", ascending =False).head()

,id,job_title,location,connection,TF-IDF_fit_score,GloVe_fit_score
5,6,aspiring human resource specialist,greater new york city area,1,0.679890,0.969614
2,3,aspiring human resource professional,raleigh durham north carolina area,44,0.781184,0.958512
45,97,aspiring human resource professional,kokomo indiana area,71,0.781184,0.958512
21,73,aspiring human resource manager seeking intern...,houston texas area,7,0.618600,0.934672
22,74,human resource professional,greater boston area,16,0.421758,0.933199


### 3.Word2Vec

In [ ]:
GoogleNews_model = api.load('word2vec-google-news-300')

In [ ]:
model_fitt_score(df, 'job_title', GoogleNews_model, 300, phrases.phrase[0], 'GoogleNews_fit_score')

In [ ]:
df.sort_values(by ='GoogleNews_fit_score', ascending = False).head()

,id,job_title,location,connection,TF-IDF_fit_score,GloVe_fit_score,GoogleNews_fit_score
45,97,aspiring human resource professional,kokomo indiana area,71,0.781184,0.958512,0.950395
2,3,aspiring human resource professional,raleigh durham north carolina area,44,0.781184,0.958512,0.950395
5,6,aspiring human resource specialist,greater new york city area,1,0.679890,0.969614,0.912262
21,73,aspiring human resource manager seeking intern...,houston texas area,7,0.618600,0.934672,0.875945
22,74,human resource professional,greater boston area,16,0.421758,0.933199,0.874494


## 4. BERT

In [ ]:
from sentence_transformers import SentenceTransformer
# build bert_base model
bert_model = SentenceTransformer("bert-base-nli-mean-tokens")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.99k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Convert job title into BERT embedded vectors
bert_job_title_embeddings =bert_model.encode(list(df.job_title))
bert_job_title_embeddings.shape

(53, 768)

In [ ]:
# convert search phrase into a bert embedded vector
bert_search_phrase_embeddings = bert_model.encode(phrases.phrase[0])
bert_search_phrase_embeddings.shape

(768,)

In [ ]:
# Calculate cosine similarity between job title and search phrase vectors
bert_cosine_similarity= []
for i in range(len(df)):
    cos_sim = 1-cosine(bert_job_title_embeddings[i], bert_search_phrase_embeddings)
    bert_cosine_similarity.append(cos_sim)


# add bert cosine_simirality column in th dataframe
df["BERT_model_fit_score"] =bert_cosine_similarity


In [ ]:
df.sort_values(by="BERT_model_fit_score", ascending=False).head()

,id,job_title,location,connection,TF-IDF_fit_score,GloVe_fit_score,GoogleNews_fit_score,BERT_model_fit_score
5,6,aspiring human resource specialist,greater new york city area,1,0.679890,0.969614,0.912262,0.955137
2,3,aspiring human resource professional,raleigh durham north carolina area,44,0.781184,0.958512,0.950395,0.948828
45,97,aspiring human resource professional,kokomo indiana area,71,0.781184,0.958512,0.950395,0.948828
30,82,aspiring human resource professional energeti...,austin texas area,174,0.379605,0.895589,0.827266,0.867910
47,99,seeking human resource position,las vegas nevada area,48,0.289466,0.869983,0.728513,0.849294


In [ ]:
df.sort_values(by="BERT_model_fit_score", ascending=False).tail()

,id,job_title,location,connection,TF-IDF_fit_score,GloVe_fit_score,GoogleNews_fit_score,BERT_model_fit_score
44,96,student indiana university kokomo business ma...,lafayette indiana,19,0.0,0.517594,0.314945,0.230208
50,102,business intelligence analytics not tech,greater new york city area,49,0.0,0.662951,0.344934,0.198931
41,93,admission representative community medical cen...,long beach california,9,0.0,0.664062,0.262623,0.181031
33,85,rrp brand portfolio executive not tech japan t...,greater philadelphia area,500,0.0,0.579212,0.271064,0.147233
35,87,bachelor science biology victoria university w...,baltimore maryland,40,0.0,0.518551,0.277453,0.136410


**Among all of the models, the BERT model gives us a high score for the top rows and a low one for unrelated titles.**

## Learning to Rank (LTR)

In [ ]:
df.head()

,id,job_title,location,connection,TF-IDF_fit_score,GloVe_fit_score,GoogleNews_fit_score,BERT_model_fit_score
0,1,ct bauer college business graduate magna cum...,houston texas,85,0.254004,0.627218,0.592438,0.557126
1,2,native english teacher tech english program korea,kanada,500,0.000000,0.650306,0.283308,0.408490
2,3,aspiring human resource professional,raleigh durham north carolina area,44,0.781184,0.958512,0.950395,0.948828
3,4,people development coordinator not tech,denton texas,500,0.000000,0.734886,0.358650,0.318839
4,5,advisory board member celal bayar university,i̇zmir türkiye,500,0.000000,0.450873,0.206563,0.461810


In [ ]:
star_candidate=input("Do you want to star any candidates? Enter 'Yes' or 'No': ")

starred= []
if star_candidate.lower()=="yes":
    starred=[int(item) for item in input("Enter ids of candidates you want to star (separated by space) : ").split()]

Do you want to star any candidates? Enter 'Yes' or 'No': no


In [ ]:
df["starred_score"] =df["BERT_model_fit_score"]
for id_num in starred:
    df.loc[df['id'] == id_num, "starred_score"]=1
df.head()

,id,job_title,location,connection,TF-IDF_fit_score,GloVe_fit_score,GoogleNews_fit_score,BERT_model_fit_score,starred_score
0,1,ct bauer college business graduate magna cum...,houston texas,85,0.254004,0.627218,0.592438,0.557126,0.557126
1,2,native english teacher tech english program korea,kanada,500,0.000000,0.650306,0.283308,0.408490,0.408490
2,3,aspiring human resource professional,raleigh durham north carolina area,44,0.781184,0.958512,0.950395,0.948828,0.948828
3,4,people development coordinator not tech,denton texas,500,0.000000,0.734886,0.358650,0.318839,0.318839
4,5,advisory board member celal bayar university,i̇zmir türkiye,500,0.000000,0.450873,0.206563,0.461810,0.461810


In [ ]:
import torch
import torch.nn as nn

# Set a seed value
seed_value= 12321

# 1. Set `PYTHONHASHSEED` environment variable at a fixed value
os.environ['PYTHONHASHSEED']=str(seed_value)

# 2. Set `python` built-in pseudo-random generator at a fixed value
random.seed(seed_value)

# 3. Set `numpy` pseudo-random generator at a fixed value
np.random.seed(seed_value)

# 4. Set `torch` pseudo-random generator at a fixed value
torch.manual_seed(seed_value)

In [ ]:
class RankNet(nn.Module):

    def __init__(self, num_feature):
        super(RankNet, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(num_feature, 512),
            nn.Dropout(0.5),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.Dropout(0.5),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid())

        self.output_sig = nn.Sigmoid()

    def forward(self, input_1, input_2):
        s1 = self.model(input_1)
        s2 = self.model(input_2)
        out = self.output_sig(s1-s2)
        return out

    def predict(self, input_):
        s = self.model(input_)
        return s

In [ ]:
random_row_1 = df.sample(n=100, replace=True)
random_row_2 =df.sample(n=100, replace=True)
job_title_list_ranknet1 = list(random_row_1["job_title"])
job_title_list_ranknet2 = list(random_row_2["job_title"])
doc_1 =bert_model.encode(job_title_list_ranknet1)
doc_2 =bert_model.encode(job_title_list_ranknet2)
doc_1 =torch.from_numpy(doc_1).float()
doc_2 = torch.from_numpy(doc_2).float()

In [ ]:
y_1 = list(random_row_1['starred_score'])
y_2 = list(random_row_2['starred_score'])
y = torch.tensor([1.0 if y1_i>y2_i else 0.5 if y1_i==y2_i else 0.0 for y1_i, y2_i in zip(y_1, y_2)]).float()

y = y.unsqueeze(1)

In [ ]:
def train_model(optim, lr_list, epoch, patience):
    # track the training loss as the model trains
    losses= []

    model =RankNet(num_feature=768)
    loss_fun= torch.nn.BCELoss()

    for lr in lr_list:
        if optim =="SGD":
            optimizer =torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
        elif optim=="Adam":
            optimizer =torch.optim.Adam(model.parameters(), lr=lr)
        print("lr: ", lr, "optimizer: ", optim)
        epoch = epoch
        train_losses  =[]
        valid_losses = []

        for i in range(1, epoch+1):
            model.zero_grad()
            y_pred =model(doc_1, doc_2)
            loss=loss_fun(y_pred, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

            if i %100 ==0:
                print("Epoch{}, loss: {}".format(i, loss.item()))

        print("="*100)
    return model, losses

In [ ]:
model, train_loss= train_model(optim="SGD",
                              lr_list=[0.1,0.01,0.001,0.0001,0.00001],
                              epoch=1000,
                              patience=20)

lr:  0.1 optimizer:  SGD
Epoch100, loss: 0.5030467510223389
Epoch200, loss: 0.49223917722702026
Epoch300, loss: 0.4902014434337616
Epoch400, loss: 0.49691540002822876
Epoch500, loss: 0.4850122332572937
Epoch600, loss: 0.4900193512439728
Epoch700, loss: 0.4911485016345978
Epoch800, loss: 0.4808211922645569
Epoch900, loss: 0.48683151602745056
Epoch1000, loss: 0.48616209626197815
lr:  0.01 optimizer:  SGD
Epoch100, loss: 0.487465500831604
Epoch200, loss: 0.48664215207099915
Epoch300, loss: 0.4865703880786896
Epoch400, loss: 0.4903468191623688
Epoch500, loss: 0.49113383889198303
Epoch600, loss: 0.48764845728874207
Epoch700, loss: 0.4897323250770569
Epoch800, loss: 0.49139440059661865
Epoch900, loss: 0.49100181460380554
Epoch1000, loss: 0.4865650236606598
lr:  0.001 optimizer:  SGD
Epoch100, loss: 0.4892933666706085
Epoch200, loss: 0.48700904846191406
Epoch300, loss: 0.4943595826625824
Epoch400, loss: 0.4896918833255768
Epoch500, loss: 0.4866357445716858
Epoch600, loss: 0.49233993887901306


In [ ]:
model, train_loss =train_model(optim="SGD", lr_list=[0.01],
                              epoch=2000,
                              patience=20)

lr:  0.01 optimizer:  SGD
Epoch100, loss: 0.5141320824623108
Epoch200, loss: 0.5019828081130981
Epoch300, loss: 0.49964410066604614
Epoch400, loss: 0.49616187810897827
Epoch500, loss: 0.49630677700042725
Epoch600, loss: 0.48644933104515076
Epoch700, loss: 0.48735514283180237
Epoch800, loss: 0.49484193325042725
Epoch900, loss: 0.4893605709075928
Epoch1000, loss: 0.49076738953590393
Epoch1100, loss: 0.48449066281318665
Epoch1200, loss: 0.4910631477832794
Epoch1300, loss: 0.4890429675579071
Epoch1400, loss: 0.49063533544540405
Epoch1500, loss: 0.4906488358974457
Epoch1600, loss: 0.4918534755706787
Epoch1700, loss: 0.4934749901294708
Epoch1800, loss: 0.487568199634552
Epoch1900, loss: 0.48633861541748047
Epoch2000, loss: 0.4893833100795746


In [ ]:
model, train_loss =train_model(optim="Adam",
                              lr_list=[0.1,0.01,0.001,0.0001,0.00001],
                              epoch=1000,
                              patience=20)

lr:  0.1 optimizer:  Adam
Epoch100, loss: 0.6931471228599548
Epoch200, loss: 0.6993482708930969
Epoch300, loss: 0.6931471228599548
Epoch400, loss: 0.6931471228599548
Epoch500, loss: 0.6893482804298401
Epoch600, loss: 0.7055494785308838
Epoch700, loss: 0.6931471228599548
Epoch800, loss: 0.7117506265640259
Epoch900, loss: 0.6931471228599548
Epoch1000, loss: 0.6931471228599548
lr:  0.01 optimizer:  Adam
Epoch100, loss: 0.6931471228599548
Epoch200, loss: 0.6931471228599548
Epoch300, loss: 0.6931471228599548
Epoch400, loss: 0.6931471228599548
Epoch500, loss: 0.6931471228599548
Epoch600, loss: 0.6931471228599548
Epoch700, loss: 0.6931471228599548
Epoch800, loss: 0.6931471228599548
Epoch900, loss: 0.6931471228599548
Epoch1000, loss: 0.6931471228599548
lr:  0.001 optimizer:  Adam
Epoch100, loss: 0.6931471228599548
Epoch200, loss: 0.6931471228599548
Epoch300, loss: 0.6931471228599548
Epoch400, loss: 0.6931471228599548
Epoch500, loss: 0.6931471228599548
Epoch600, loss: 0.6931471228599548
Epoch70

In [ ]:
class RankNet(nn.Module):

    def __init__(self, num_feature):
        super(RankNet, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(num_feature, num_feature*2),
            nn.Dropout(0.2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(num_feature*2, num_feature *4),
            nn.Dropout(0.2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(num_feature*4,1),
            nn.Sigmoid())
        self.output_sig =nn.Sigmoid()

    def forward(self, input_1, input_2):
        s1 =self.model(input_1)
        s2 =self.model(input_2)
        out =self.output_sig(s1-s2)
        return out

    def predict(self, input_):
        s = self.model(input_)
        return s

In [ ]:
model, train_loss =train_model(optim="SGD",
                              lr_list=[0.01],
                              epoch=2000,
                              patience=20)

lr:  0.01 optimizer:  SGD
Epoch100, loss: 0.5049774050712585
Epoch200, loss: 0.49336501955986023
Epoch300, loss: 0.4924640953540802
Epoch400, loss: 0.4901599884033203
Epoch500, loss: 0.4881107211112976
Epoch600, loss: 0.48964259028434753
Epoch700, loss: 0.48760509490966797
Epoch800, loss: 0.487868070602417
Epoch900, loss: 0.4876951575279236
Epoch1000, loss: 0.4908272624015808
Epoch1100, loss: 0.4900563359260559
Epoch1200, loss: 0.4884736239910126
Epoch1300, loss: 0.4926424026489258
Epoch1400, loss: 0.4862217605113983
Epoch1500, loss: 0.48733484745025635
Epoch1600, loss: 0.4875853359699249
Epoch1700, loss: 0.4908725321292877
Epoch1800, loss: 0.48595133423805237
Epoch1900, loss: 0.48640117049217224
Epoch2000, loss: 0.48670056462287903


In [ ]:
pred_score = []
for i in range(len(df)):
    embedding =bert_model.encode([df["job_title"][i]])
    embedding_tensor = torch.from_numpy(embedding).float()
    pred = round(model.predict(embedding_tensor).detach().numpy().sum(),2)
    pred_score.append(pred)



In [ ]:
df["RankNet_score"] =pred_score
df.sort_values(by="RankNet_score", ascending =False).head()

,id,job_title,location,connection,TF-IDF_fit_score,GloVe_fit_score,GoogleNews_fit_score,BERT_model_fit_score,starred_score,RankNet_score
52,104,director administration excellence logtechng,katy texas,500,0.000000,0.665444,0.191680,0.621203,0.621203,1.0
21,73,aspiring human resource manager seeking intern...,houston texas area,7,0.618600,0.934672,0.875945,0.752413,0.752413,1.0
24,76,aspiring human resource professional passiona...,new york new york,212,0.259969,0.917865,0.741262,0.765864,0.765864,1.0
27,79,liberal art major aspiring human resource ana...,baton rouge louisiana area,7,0.354417,0.850251,0.731783,0.557962,0.557962,1.0
30,82,aspiring human resource professional energeti...,austin texas area,174,0.379605,0.895589,0.827266,0.867910,0.867910,1.0


## CONCLUSION

`BERT` Model was one of the best model to find similarity between our data and the targeted phrase. As for the Ranking Model I ran `RankNet` model on the data and the best Loss score was 48%